In [ ]:
 from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

repo = Path("/content/facial-visual-profile")
source = Path("/content/drive/MyDrive/FacialVisualProfile/data")

required = (
    "img_align_celeba.zip",
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
)

for name in required:
    path = source / name
    if not path.is_file():
        raise FileNotFoundError(f"Missing source file: {path}")
    print(f"Found: {name}")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/DeweyYip/facial-visual-profile.git",
            str(repo),
        ],
        check=True,
    )

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Working directory:", Path.cwd())
print(
    "Git commit:",
    subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], text=True
    ).strip(),
)

In [ ]:
import shutil
import subprocess

raw = repo / "data/raw"
raw.mkdir(parents=True, exist_ok=True)

for name in required:
    src = source / name
    dst = raw / name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print(f"Copying: {name}", flush=True)
        shutil.copy2(src, dst)

image_dir = raw / "img_align_celeba"
image_count = sum(1 for _ in image_dir.glob("*.jpg"))

if image_count != 202599:
    print("Extracting images...", flush=True)
    subprocess.run(
        ["unzip", "-oq", str(raw / "img_align_celeba.zip"), "-d", str(raw)],
        check=True,
    )

image_count = sum(1 for _ in image_dir.glob("*.jpg"))
assert image_count == 202599, f"Unexpected image count: {image_count}"

print("Extracted images:", image_count)
print("Local data preparation passed.")

In [ ]:
import sys
from pathlib import Path
from importlib import metadata

print("Python:", sys.version.split()[0])

for package in ("torch", "torchvision", "facenet-pytorch", "mtcnn"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

repo = Path("/content/facial-visual-profile")
image_dir = repo / "data/raw/img_align_celeba"

print("Repository exists:", repo.is_dir())
print("First image exists:", (image_dir / "000001.jpg").is_file())
print(
    "Aligned landmarks exist:",
    (repo / "data/raw/list_landmarks_align_celeba.txt").is_file(),
)

Prepare the dataset

In [ ]:
%pip install --no-deps facenet-pytorch==2.6.0

check if it works

In [ ]:
print("Original landmark dtype:", landmarks.dtype)

boxes = np.asarray(boxes, dtype=np.float32)
probabilities = np.asarray(probabilities, dtype=np.float32)
landmarks = np.asarray(landmarks, dtype=np.float32)

assert landmarks.shape == (len(boxes), 5, 2)
assert np.isfinite(boxes).all()
assert np.isfinite(probabilities).all()
assert np.isfinite(landmarks).all()

print("Faces detected:", len(boxes))
print("Detection probabilities:", probabilities.tolist())
print("Landmark shape:", landmarks.shape)
print("Predicted landmarks:", landmarks.tolist())
print("Single-image detection check passed.")

Test on the image

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path("/content/facial-visual-profile/data/raw")
name = "000001.jpg"

gt = None
with (root / "list_landmarks_align_celeba.txt").open(
    encoding="utf-8-sig"
) as f:
    next(f)
    columns = next(f).split()
    for line in f:
        filename, *values = line.split()
        if filename == name:
            gt = np.asarray(values, dtype=np.float32).reshape(5, 2)
            break

if gt is None:
    raise ValueError(f"Missing GT landmarks: {name}")

assert len(landmarks) == 1, "This inspection expects exactly one face."
predicted = landmarks[0]

print("GT coordinate columns:", columns)
print("GT landmarks:", gt.tolist())

errors = np.linalg.norm(predicted - gt, axis=1)
eye_distance = np.linalg.norm(gt[0] - gt[1])
assert eye_distance > 0
nme = errors.mean() / eye_distance

print("Errors per point (pixels):", errors.round(3).tolist())
print(f"Mean point error: {errors.mean():.3f} pixels")
print(f"NME (GT eye-distance normalization): {nme:.4f} ({nme:.2%})")

fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(image)

ax.scatter(
    gt[:, 0], gt[:, 1],
    marker="o", s=70, facecolors="none",
    edgecolors="lime", linewidths=2, label="GT",
)
ax.scatter(
    predicted[:, 0], predicted[:, 1],
    marker="x", s=60, color="red",
    linewidths=2, label="MTCNN",
)

for index, (reference, estimate) in enumerate(zip(gt, predicted), start=1):
    ax.plot(
        [reference[0], estimate[0]],
        [reference[1], estimate[1]],
        color="yellow", linewidth=1,
    )
    ax.annotate(
        str(index), reference + np.array([4, -4]),
        color="yellow",
    )

ax.set_title(f"{name} | NME: {nme:.2%}")
ax.legend()
ax.axis("off")
plt.tight_layout()
plt.show()

Test on 50 images

In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

root = Path("/content/facial-visual-profile/data/raw")

with (root / "list_eval_partition.txt").open() as f:
    train_names = [
        name
        for line in f
        for name, split in [line.split()]
        if split == "0"
    ]

sample_names = random.Random(2026).sample(train_names, 50)
sample_set = set(sample_names)
gt_by_name = {}

with (root / "list_landmarks_align_celeba.txt").open(
    encoding="utf-8-sig"
) as f:
    next(f)
    next(f)
    for line in f:
        name, *values = line.split()
        if name in sample_set:
            gt_by_name[name] = np.asarray(
                values, dtype=np.float32
            ).reshape(5, 2)

results = []

for name in sample_names:
    with Image.open(root / "img_align_celeba" / name) as source:
        rgb = source.convert("RGB")

    with torch.inference_mode():
        boxes, scores, points = detector.detect(rgb, landmarks=True)

    result = {
        "name": name,
        "image": rgb,
        "gt": gt_by_name[name],
        "faces": 0 if boxes is None else len(boxes),
        "status": "no_face",
        "predicted": None,
        "nme": None,
    }

    if result["faces"] > 0:
        scores = np.asarray(scores, dtype=np.float32)
        points = np.asarray(points, dtype=np.float32)

        if (
            points.shape != (result["faces"], 5, 2)
            or not np.isfinite(scores).all()
            or not np.isfinite(points).all()
        ):
            result["status"] = "invalid_output"
        else:
            selected = int(np.argmax(scores))
            predicted = points[selected]
            gt = result["gt"]
            eye_distance = np.linalg.norm(gt[0] - gt[1])
            assert eye_distance > 0, f"Invalid GT eye distance: {name}"

            result["status"] = "ok"
            result["predicted"] = predicted
            result["nme"] = float(
                np.linalg.norm(predicted - gt, axis=1).mean()
                / eye_distance
            )

    results.append(result)

for start in range(0, len(results), 10):
    fig, axes = plt.subplots(2, 5, figsize=(15, 8))
    for ax, result in zip(axes.flat, results[start:start + 10]):
        ax.imshow(result["image"])
        gt = result["gt"]
        ax.scatter(
            gt[:, 0], gt[:, 1], s=40,
            facecolors="none", edgecolors="lime",
        )

        predicted = result["predicted"]
        if predicted is not None:
            ax.scatter(
                predicted[:, 0], predicted[:, 1],
                marker="x", s=30, color="red",
            )
            detail = f"NME {result['nme']:.1%}"
        else:
            detail = result["status"]

        ax.set_title(
            f"{result['name']}\n"
            f"{detail} | faces: {result['faces']}",
            fontsize=9,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

counts = Counter(result["status"] for result in results)
valid_results = [r for r in results if r["nme"] is not None]
nmes = np.asarray([r["nme"] for r in valid_results])

print("Samples:", len(results))
print("Status counts:", dict(counts))
print("Multiple-face samples:", sum(r["faces"] > 1 for r in results))
print(
    "Detection failure rate:",
    f"{sum(r['status'] != 'ok' for r in results) / len(results):.1%}",
)

if len(nmes):
    print(f"Mean NME (successful detections): {nmes.mean():.2%}")
    print(f"Median NME (successful detections): {np.median(nmes):.2%}")
    print("Five highest-NME samples:")
    for result in sorted(
        valid_results, key=lambda r: r["nme"], reverse=True
    )[:5]:
        print(f"  {result['name']}: {result['nme']:.2%}")

In [ ]:
for result in results:
    if result["faces"] <= 1:
        continue

    print("Multiple-face sample:", result["name"])

    with torch.inference_mode():
        boxes, scores, points = detector.detect(
            result["image"], landmarks=True
        )

    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    points = np.asarray(points, dtype=np.float32)
    selected = int(np.argmax(scores))

    fig, ax = plt.subplots(figsize=(7, 8))
    ax.imshow(result["image"])

    for index, (box, score, face_points) in enumerate(
        zip(boxes, scores, points)
    ):
        color = "red" if index == selected else "cyan"
        x1, y1, x2, y2 = box
        ax.add_patch(plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor=color, linewidth=2,
        ))
        ax.scatter(
            face_points[:, 0], face_points[:, 1],
            marker="x", color=color,
        )
        ax.text(
            x1, y1,
            f"Face {index}: {score:.6f}",
            color=color, fontsize=10,
        )

    gt = result["gt"]
    ax.scatter(
        gt[:, 0], gt[:, 1],
        s=70, facecolors="none", edgecolors="lime",
        linewidths=2, label="GT",
    )
    ax.set_title(
        f"{result['name']} | selected face: {selected}\n"
        "Red: selected | Cyan: other detections"
    )
    ax.legend()
    ax.axis("off")
    plt.tight_layout()
    plt.show()

We don't run MTCNN on our local environment at this stage. Running on colab for test is enough

In [ ]:
import os
import subprocess
import sys

os.chdir("/content/facial-visual-profile")

subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
command = [
    sys.executable, "-m", "scripts.cache_mtcnn",
    "--device", "cuda",
    "--limit", "50",
    "--output", "data/cache/mtcnn_smoke.sqlite",
]

print("First run", flush=True)
subprocess.run(command, check=True)

print("Second run: checking resume behavior", flush=True)
subprocess.run(command, check=True)

In [ ]:
from pathlib import Path
import sqlite3
import subprocess
import sys

repo = Path("/content/facial-visual-profile")
cache = repo / "data/cache/mtcnn_smoke.sqlite"

assert cache.is_file(), f"Missing cache: {cache}"

with sqlite3.connect(f"{cache.as_uri()}?mode=ro", uri=True) as connection:
    count = connection.execute(
        "SELECT COUNT(*) FROM detections"
    ).fetchone()[0]

print("Cached records:", count, flush=True)

result = subprocess.run(
    [
        sys.executable, "-m", "scripts.cache_mtcnn",
        "--device", "cuda",
        "--limit", "50",
        "--output", str(cache),
    ],
    cwd=repo,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

In [ ]:
from pathlib import Path
import sqlite3
import subprocess
import sys
import time

repo = Path("/content/facial-visual-profile")
local_cache = repo / "data/cache/mtcnn_smoke.sqlite"
drive_root = Path("/content/drive/MyDrive")

assert drive_root.is_dir(), "Mount Google Drive first."

backup_dir = drive_root / "FacialVisualProfile/cache"
backup_dir.mkdir(parents=True, exist_ok=True)
drive_cache = backup_dir / "mtcnn_clean.sqlite"


def backup_database(source, destination):
    temporary = destination.with_name(destination.name + ".tmp")
    with sqlite3.connect(
        f"{source.as_uri()}?mode=ro", uri=True
    ) as source_db:
        with sqlite3.connect(temporary) as destination_db:
            source_db.backup(destination_db)
    temporary.replace(destination)


# Restore the persistent backup after a runtime reset.
if not local_cache.exists() and drive_cache.exists():
    local_cache.parent.mkdir(parents=True, exist_ok=True)
    backup_database(drive_cache, local_cache)
    print("Restored cache from Google Drive.", flush=True)

assert local_cache.is_file(), "Local cache is missing."

if not drive_cache.exists():
    backup_database(local_cache, drive_cache)
    print("Initial backup saved.", flush=True)

total = 202599
batch_size = 5000
started = time.perf_counter()

for limit in range(batch_size, total + batch_size, batch_size):
    limit = min(limit, total)
    print(f"\nProcessing up to image {limit}/{total}", flush=True)

    result = subprocess.run(
        [
            sys.executable, "-m", "scripts.cache_mtcnn",
            "--device", "cuda",
            "--limit", str(limit),
            "--output", str(local_cache),
        ],
        cwd=repo,
        capture_output=True,
        text=True,
    )

    print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        result.check_returncode()

    backup_database(local_cache, drive_cache)
    print("Google Drive backup updated.", flush=True)

with sqlite3.connect(
    f"{local_cache.as_uri()}?mode=ro", uri=True
) as connection:
    count = connection.execute(
        "SELECT COUNT(*) FROM detections"
    ).fetchone()[0]

assert count == total, f"Unexpected record count: {count}"
print(f"\nTotal cached: {count}", flush=True)
print(f"Elapsed minutes: {(time.perf_counter() - started) / 60:.1f}")
print("Persistent cache:", drive_cache)

We have ran the MTCNN on every images in our dataset
Let's overview of the output

In [ ]:
import json
import sqlite3
from collections import Counter
from pathlib import Path

cache = Path(
    "/content/facial-visual-profile/data/cache/mtcnn_smoke.sqlite"
)
assert cache.is_file(), f"Missing cache: {cache}"

counts = Counter()
multiple_faces = 0

with sqlite3.connect(f"{cache.as_uri()}?mode=ro", uri=True) as connection:
    for (payload,) in connection.execute("SELECT result FROM detections"):
        result = json.loads(payload)
        counts[result["status"]] += 1
        multiple_faces += result["face_count"] > 1

total = sum(counts.values())
failed = total - counts["ok"]

print("Total records:", total)
print("Status counts:", dict(counts))
print("Multiple-face images:", multiple_faces)
print(f"Detection failure rate: {failed / total:.2%}")

Save the results

In [ ]:
import json
import sqlite3
from collections import Counter
from pathlib import Path

import numpy as np

root = Path("/content/facial-visual-profile/data/raw")
cache = Path(
    "/content/facial-visual-profile/data/cache/mtcnn_smoke.sqlite"
)
drive_root = Path("/content/drive/MyDrive")
assert drive_root.is_dir(), "Mount Google Drive first."

gt_by_name = {}
with (root / "list_landmarks_align_celeba.txt").open(
    encoding="utf-8-sig"
) as f:
    next(f)
    next(f)
    for line in f:
        name, *values = line.split()
        gt_by_name[name] = np.asarray(
            values, dtype=np.float64
        ).reshape(5, 2)

report = {
    "normalization": "GT inter-eye distance",
    "scope": "Training and validation only",
    "splits": {},
}

with sqlite3.connect(f"{cache.as_uri()}?mode=ro", uri=True) as connection:
    for split_name, split_code in (("train", "0"), ("valid", "1")):
        statuses = Counter()
        nmes = []
        multiple_faces = 0
        invalid_gt = 0

        rows = connection.execute(
            "SELECT name, result FROM detections WHERE split = ?",
            (split_code,),
        )

        for name, payload in rows:
            result = json.loads(payload)
            statuses[result["status"]] += 1
            multiple_faces += result["face_count"] > 1

            if result["status"] != "ok":
                continue

            gt = gt_by_name[name]
            eye_distance = np.linalg.norm(gt[0] - gt[1])
            if not np.isfinite(gt).all() or eye_distance <= 0:
                invalid_gt += 1
                continue

            predicted = np.asarray(
                result["landmarks"], dtype=np.float64
            )
            assert predicted.shape == (5, 2)
            assert np.isfinite(predicted).all()

            nmes.append(float(
                np.linalg.norm(predicted - gt, axis=1).mean()
                / eye_distance
            ))

        total = sum(statuses.values())
        values = np.asarray(nmes)
        assert values.size > 0, f"No evaluable detections: {split_name}"

        stats = {
            "total": total,
            "status_counts": dict(statuses),
            "multiple_face_images": multiple_faces,
            "failure_rate": (total - statuses["ok"]) / total,
            "invalid_gt_count": invalid_gt,
            "nme_sample_count": len(nmes),
            "mean_nme": float(values.mean()),
            "median_nme": float(np.median(values)),
            "p95_nme": float(np.percentile(values, 95)),
        }
        report["splits"][split_name] = stats

        print(f"\nSplit: {split_name}")
        print("Images:", total)
        print("Status counts:", dict(statuses))
        print("Multiple-face images:", multiple_faces)
        print("Invalid GT count:", invalid_gt)
        print(f"Mean NME: {stats['mean_nme']:.2%}")
        print(f"Median NME: {stats['median_nme']:.2%}")
        print(f"95th percentile NME: {stats['p95_nme']:.2%}")

output = drive_root / "FacialVisualProfile/cache/mtcnn_clean_quality.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print("\nSaved:", output)